# Mathematical and Statistical Foundations

This notebook implements the first module of the quant curriculum: probability sampling, random walks, geometric Brownian motion, Monte Carlo option pricing, and basic time-series diagnostics.  The checks are intentionally written like unit tests because trading research needs reproducible evidence, not vibes.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent.parent
sys.path.insert(0, str(repo_root / 'src'))

import numpy as np

from quant_lab.foundations import (
    adf_stationarity_test,
    arima_forecast,
    autocorrelation,
    black_scholes_price,
    geometric_brownian_motion,
    monte_carlo_european_option,
    random_walk,
    sample_bernoulli,
    sample_lognormal,
    sample_normal,
    sample_poisson,
    sample_student_t,
)


## 1. Probability distributions

Financial models often start with distributional assumptions. Normal returns are mathematically convenient, lognormal prices keep prices positive, Student-t distributions model fat tails, and Bernoulli/Poisson variables model binary events and event counts.

In [ ]:
normal = sample_normal(50_000, mean=1.0, std=2.0, seed=1)
lognormal = sample_lognormal(10_000, mean=0.0, sigma=0.25, seed=1)
student_t = sample_student_t(10_000, df=5, seed=1)
bernoulli = sample_bernoulli(50_000, p=0.35, seed=1)
poisson = sample_poisson(50_000, lam=3.0, seed=1)

assert abs(normal.mean() - 1.0) < 0.03
assert abs(normal.std(ddof=1) - 2.0) < 0.03
assert np.all(lognormal > 0)
assert abs(bernoulli.mean() - 0.35) < 0.01
assert abs(poisson.mean() - 3.0) < 0.04

{
    'normal_mean': float(normal.mean()),
    'normal_std': float(normal.std(ddof=1)),
    'student_t_empirical_kurtosis_hint': float(np.mean(((student_t - student_t.mean()) / student_t.std()) ** 4)),
    'bernoulli_mean': float(bernoulli.mean()),
    'poisson_mean': float(poisson.mean()),
}


## 2. Random walks and geometric Brownian motion

A random walk is an additive process. GBM is a multiplicative process commonly used for simple equity/option modelling because it keeps simulated prices positive.

In [ ]:
walk = random_walk(steps=252, start=0.0, drift=0.001, volatility=0.02, seed=2)
gbm = geometric_brownian_motion(s0=100, mu=0.06, sigma=0.20, years=1.0, steps=252, paths=2_000, seed=2)

assert walk.shape == (252,)
assert gbm.shape == (2_000, 253)
assert np.all(gbm > 0)

terminal_mean = gbm[:, -1].mean()
theoretical_mean = 100 * np.exp(0.06)
assert abs(terminal_mean - theoretical_mean) / theoretical_mean < 0.03

{'gbm_terminal_mean': float(terminal_mean), 'theoretical_mean': float(theoretical_mean)}


## 3. Monte Carlo option pricing

Under risk-neutral GBM, the discounted expected payoff of a European option should converge toward the Black-Scholes price. This is a first example of simulation-based pricing and convergence checking.

In [ ]:
mc_call = monte_carlo_european_option(s0=100, strike=100, rate=0.03, sigma=0.20, years=1.0, paths=80_000, option_type='call', seed=3)
bs_call = black_scholes_price(s0=100, strike=100, rate=0.03, sigma=0.20, years=1.0, option_type='call')

assert abs(mc_call.price - bs_call) < 3 * mc_call.standard_error + 0.15

{'monte_carlo_call': mc_call.price, 'standard_error': mc_call.standard_error, 'black_scholes_call': bs_call}


## 4. Time-series diagnostics

Autocorrelation measures persistence. Stationarity tests such as ADF help decide whether a series behaves more like a stable process or a drifting random walk. ARIMA-style forecasts are useful baselines, not magic.

In [ ]:
persistent = np.arange(200, dtype=float)
noise = sample_normal(200, seed=4)

assert autocorrelation(persistent, lag=1) > 0.95
assert abs(autocorrelation(noise, lag=1)) < 0.25

stationarity = adf_stationarity_test(noise)
forecast_status = 'not_run'
try:
    forecast = arima_forecast(noise, order=(1, 0, 0), steps=3)
    assert forecast.shape == (3,)
    forecast_status = 'ok'
except RuntimeError:
    forecast_status = 'statsmodels_unavailable'

{'noise_autocorr_lag1': autocorrelation(noise, lag=1), 'adf': stationarity, 'arima_forecast_status': forecast_status}


## Lessons and limitations

- Distributional assumptions are useful but dangerous if treated as truth.
- Monte Carlo needs convergence checks and standard errors.
- Time-series models must be evaluated out of sample.
- These primitives become trading-relevant only when combined with realistic costs, slippage, position limits, and walk-forward validation.